# 🚀 Week 7 — Production-Level System Design
> **Goal:** Make your AI system ready for real traffic — caching, logging, rate limiting, async pipelines.

---
### What you will learn
1. Response caching (skip the API call if you already know the answer)
2. Structured logging (know what your AI actually does in production)
3. Rate limiting (don't get API-banned)
4. Async concurrency (handle many requests at once)
5. Cost tracking
6. Retry logic with exponential backoff
7. Full production-grade agent wrapper

## 1 — Setup

In [ ]:
%pip install pydantic-ai python-dotenv groq rich --quiet

In [ ]:
import os, sys, time
sys.path.insert(0, ".")
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY!"

from cache import ResponseCache
from logger import AILogger, RateLimiter, estimate_cost
print("Production modules loaded.")

## 2 — Response caching

In [ ]:
from pydantic_ai import Agent
from cache import ResponseCache
import time

cache = ResponseCache(ttl_seconds=300)
agent = Agent(
    model="groq:llama-3.1-8b-instant",
    system_prompt="Answer concisely.",
)

def cached_run(prompt: str) -> str:
    """Run agent with cache — skips LLM if response already cached."""
    key = cache.make_key("groq:llama-3.1-8b-instant", prompt)

    hit = cache.get(key)
    if hit is not None:
        print(f"  [CACHE HIT] Returning cached response.")
        return hit

    print(f"  [CACHE MISS] Calling LLM...")
    start = time.perf_counter()
    result = agent.run_sync(prompt)
    elapsed_ms = (time.perf_counter() - start) * 1000

    cache.set(key, result.data)
    print(f"  [STORED] Response cached in {elapsed_ms:.0f}ms")
    return result.data


# First call — hits the LLM
prompt = "What is the capital of France?"
print("Call 1:")
ans1 = cached_run(prompt)
print(f"  Answer: {ans1}")

# Second call — served from cache (instant!)
print("\nCall 2 (same prompt):")
ans2 = cached_run(prompt)
print(f"  Answer: {ans2}")

print(f"\nCache stats: {cache.stats()}")

## 3 — Structured logging

In [ ]:
from logger import AILogger
from pydantic_ai import Agent
import time

ai_logger = AILogger(log_file="ai_calls.jsonl", console=True)
agent = Agent(
    model="groq:llama-3.1-8b-instant",
    system_prompt="Answer concisely.",
)

def logged_run(prompt: str):
    """Run agent with full logging."""
    start = time.perf_counter()
    error = None
    result_data = None

    try:
        result = agent.run_sync(prompt)
        result_data = result.data
        # PydanticAI Usage object
        usage_dict = {
            "prompt_tokens": result.usage().request_tokens or 0,
            "completion_tokens": result.usage().response_tokens or 0,
            "total_tokens": result.usage().total_tokens or 0,
        }
    except Exception as e:
        error = str(e)
        usage_dict = None

    latency_ms = (time.perf_counter() - start) * 1000

    ai_logger.log(
        model="groq:llama-3.1-8b-instant",
        prompt=prompt,
        response=result_data,
        latency_ms=latency_ms,
        usage=usage_dict,
        error=error,
    )

    return result_data


# Run a few calls
logged_run("What is machine learning?")
logged_run("Name 3 Python web frameworks.")
logged_run("What is 2 + 2?")

print(f"\nSession stats: {ai_logger.session_stats()}")

## 4 — Rate limiting

In [ ]:
import asyncio
from logger import RateLimiter
from pydantic_ai import Agent
import time

rate_limiter = RateLimiter(requests_per_minute=20)  # max 20 req/min
agent = Agent(
    model="groq:llama-3.1-8b-instant",
    system_prompt="Answer in one word.",
)

async def rate_limited_ask(question: str) -> str:
    await rate_limiter.acquire()  # waits if needed
    result = await agent.run(question)
    return result.data


questions = ["Capital of France?", "Capital of Japan?", "Capital of Germany?"]

start = time.perf_counter()
# These will be spaced ~3 seconds apart (60s / 20rpm)
# (reduced to 3 for demo — in prod you'd use higher RPM)
answers = await asyncio.gather(*[rate_limited_ask(q) for q in questions])
elapsed = time.perf_counter() - start

for q, a in zip(questions, answers):
    print(f"  Q: {q:<30} A: {a}")

print(f"\nTotal time: {elapsed:.2f}s (rate limiter enforced spacing)")

## 5 — Retry logic with exponential backoff

In [ ]:
import asyncio
import random
from pydantic_ai import Agent
from pydantic_ai.exceptions import UnexpectedModelBehavior

async def retry_run(
    agent: Agent,
    prompt: str,
    max_retries: int = 3,
    base_delay: float = 1.0,
):
    """
    Run an agent with exponential backoff retry.
    Delays: 1s, 2s, 4s (+ jitter to prevent thundering herd).
    """
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            result = await agent.run(prompt)
            if attempt > 0:
                print(f"  Succeeded on attempt {attempt + 1}")
            return result.data
        except Exception as e:
            last_error = e
            if attempt < max_retries:
                delay = base_delay * (2 ** attempt) + random.uniform(0, 0.5)  # jitter
                print(f"  Attempt {attempt + 1} failed ({type(e).__name__}). Retrying in {delay:.1f}s...")
                await asyncio.sleep(delay)
            else:
                print(f"  All {max_retries + 1} attempts failed.")
                raise


retry_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    system_prompt="Answer concisely.",
)

answer = await retry_run(retry_agent, "What is async programming in Python?")
print(f"Answer: {answer[:200]}")

## 6 — Async concurrency (processing many requests)

In [ ]:
import asyncio
import time
from pydantic import BaseModel, Field
from typing import List
from pydantic_ai import Agent

class Sentiment(BaseModel):
    text: str
    label: str = Field(description="positive, negative, or neutral")
    score: float = Field(ge=0.0, le=1.0)

sentiment_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    result_type=Sentiment,
    system_prompt="Classify the sentiment of the given text. Return the original text.",
)

texts = [
    "I love this product! It changed my life.",
    "Terrible experience. Would not recommend.",
    "It arrived on time. Works as expected.",
    "Amazing quality, super fast shipping!",
    "Disappointed. The battery dies in 2 hours.",
]

async def analyse(text: str) -> Sentiment:
    result = await sentiment_agent.run(text)
    return result.data

# Process all texts in PARALLEL — concurrently (not sequentially)
start = time.perf_counter()
results = await asyncio.gather(*[analyse(t) for t in texts])
elapsed = time.perf_counter() - start

print(f"Analysed {len(results)} texts in {elapsed:.2f}s (parallel)")
print()
for r in results:
    emoji = "😊" if r.label == "positive" else "😞" if r.label == "negative" else "😐"
    print(f"  {emoji} [{r.label:<8} {r.score:.2f}] {r.text[:50]}")

## 7 — Production Agent Wrapper (putting it all together)

In [ ]:
import asyncio
import time
import random
from typing import TypeVar, Type, Any
from pydantic import BaseModel
from pydantic_ai import Agent
from cache import ResponseCache
from logger import AILogger, RateLimiter

T = TypeVar("T")

class ProductionAgent:
    """
    Production-ready wrapper around PydanticAI Agent.
    Adds: caching, logging, rate limiting, retry logic.
    """

    def __init__(
        self,
        agent: Agent,
        model_name: str,
        cache_ttl: int = 300,
        rpm_limit: int = 30,
        max_retries: int = 3,
        log_file: str = "production_ai.jsonl",
    ):
        self.agent = agent
        self.model_name = model_name
        self.cache = ResponseCache(ttl_seconds=cache_ttl)
        self.logger = AILogger(log_file=log_file)
        self.limiter = RateLimiter(requests_per_minute=rpm_limit)
        self.max_retries = max_retries

    async def run(self, prompt: str, use_cache: bool = True) -> Any:
        # 1. Check cache
        cache_key = self.cache.make_key(self.model_name, prompt)
        if use_cache:
            cached = self.cache.get(cache_key)
            if cached is not None:
                return cached

        # 2. Rate limit
        await self.limiter.acquire()

        # 3. Run with retry
        start = time.perf_counter()
        error_msg = None
        result_data = None

        for attempt in range(self.max_retries + 1):
            try:
                result = await self.agent.run(prompt)
                result_data = result.data

                usage_dict = {
                    "prompt_tokens": result.usage().request_tokens or 0,
                    "completion_tokens": result.usage().response_tokens or 0,
                    "total_tokens": result.usage().total_tokens or 0,
                }
                break
            except Exception as e:
                error_msg = str(e)
                usage_dict = None
                if attempt < self.max_retries:
                    delay = (2 ** attempt) + random.uniform(0, 0.3)
                    await asyncio.sleep(delay)

        latency_ms = (time.perf_counter() - start) * 1000

        # 4. Log
        self.logger.log(
            model=self.model_name,
            prompt=prompt,
            response=result_data,
            latency_ms=latency_ms,
            usage=usage_dict,
            error=error_msg,
        )

        # 5. Cache successful result
        if result_data is not None and use_cache:
            self.cache.set(cache_key, result_data)

        if error_msg and result_data is None:
            raise RuntimeError(f"Agent failed after {self.max_retries} retries: {error_msg}")

        return result_data

    def stats(self) -> dict:
        return {
            "cache": self.cache.stats(),
            "session": self.logger.session_stats(),
        }


# ── Demo ────────────────────────────────────────────────────────────────────
base_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    system_prompt="You are a helpful assistant. Answer in 1-2 sentences.",
)

prod_agent = ProductionAgent(
    agent=base_agent,
    model_name="groq:llama-3.1-8b-instant",
    cache_ttl=300,
    rpm_limit=60,
)

# First run — LLM called
r1 = await prod_agent.run("What is PydanticAI?")
print(f"Response: {r1[:100]}...")

# Second run same prompt — cached!
r2 = await prod_agent.run("What is PydanticAI?")
print(f"Cached  : {r2[:100]}...")

print(f"\nFull stats: {prod_agent.stats()}")

## Week 7 — Summary
| Feature | Tool / Pattern |
|---|---|
| Caching | `ResponseCache` — TTL-based, hash key from model+prompt |
| Logging | `AILogger` — JSONL structured log + console |
| Rate limiting | `RateLimiter` — token bucket (RPM-based) |
| Retry | Exponential backoff + jitter |
| Concurrency | `asyncio.gather()` for parallel requests |
| Cost tracking | Per-model token price × usage |
| Full wrapper | `ProductionAgent` combines all of the above |

**Next: Week 8 — Final Projects!**